In [2]:
# Quick sanity check: hand-eye calibration uses OpenCV's aruco/charuco module.
# You need opencv-contrib-python, not only the base opencv-python package.
# If ``cv2.aruco`` is missing, reinstall the contrib build in this environment.
import cv2
print(cv2.__version__)

4.11.0


In [3]:
# Numerical and filesystem utilities.
import numpy as np
import os
import sys

# Open3D is used only for optional point-cloud/TSDF visualization after the
# calibration result is computed. The core hand-eye solve does not require it.
import open3d as o3d
import cv2
from tqdm import trange

# roboticstoolbox is used here through its built-in Panda model. This is a
# Franka-specific dependency in the current notebook.
#
# ViperX replacement point:
# - Load or define a ViperX model instead of ``rtb.models.Panda()`` below.
# - Make sure the model's joint order and end-effector link match your real
#   ViperX data source.
import roboticstoolbox as rtb

# Add the repo's ``real-deployment/calibration`` parent path so local modules
# can be imported from an interactive notebook session.
parent_dir = os.path.dirname(os.getcwd())
parent_dir = os.path.dirname(parent_dir)
sys.path.append(parent_dir)

# ``HandinEyeCalibrator`` wraps ChArUco detection and ``cv2.calibrateHandEye``.
# ``read_data`` loads the folder produced by ``hand_in_eye_shooting.ipynb``.
from calibration.hand_in_eye import HandinEyeCalibrator
from calibration.utils import read_data

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Calibration：从采集数据求 `camera -> hand`

这一节读取 `hand_in_eye_shooting.ipynb` 保存的数据，并调用 `HandinEyeCalibrator.perform()` 求解相机到机器人手部的固定外参。

坐标系约定：OpenCV 的 `calibrateHandEye` 需要每一帧的 `gripper/hand -> base` 和 `target -> camera`。代码里的 `pose_list` 应该最终表示 `T_hand_to_base`。当前 notebook 里的 `tcp_to_hand()` 和 `joint_to_hand()` 都是为了把 Panda 专有数据转换成这个统一语义。

迁移到 ViperX 时，最重要的工作就是替换这两个转换函数：用 ViperX 的 FK/IK 或你的 LeRobot 状态接口，保证输出仍然是每帧的 4x4 `T_hand_to_base`。

In [ ]:
def tcp_to_hand(pose):
    """Convert a Panda TCP pose to the Panda hand-link pose.

    The calibration solver wants a consistent end-effector frame named
    gripper/hand. Some robot APIs expose a TCP pose instead. The original Panda
    workaround uses IK to recover one joint solution for that TCP pose, then FK
    to the ``panda_hand`` link.

    ViperX replacement point:
    - Do not keep this Panda IK/FK path for ViperX.
    - If your recorded pose is already your chosen ViperX hand frame, return it
      directly.
    - If it is a tool-tip/TCP frame, multiply by the fixed transform between
      TCP and your hand/camera-mount frame.
    """
    panda = rtb.models.Panda()
    joints = panda.ik_LM(pose)[0]
    hand_pose = panda.fkine(joints, end="panda_hand").A
    return hand_pose

def joint_to_hand(joints):
    """Convert Panda joint angles to a Panda hand-link pose using FK.

    This is usually the cleaner path because it starts from measured joints.
    FK means forward kinematics: joint angles -> end-effector pose.

    ViperX replacement point:
    - Replace ``rtb.models.Panda()`` with your ViperX model.
    - Replace ``end="panda_hand"`` with the ViperX end-effector link you choose
      as the hand frame.
    - Check units carefully. roboticstoolbox normally expects radians.
    """
    panda = rtb.models.Panda()
    hand_pose = panda.fkine(joints, end="panda_hand").A
    return hand_pose

# Read the dataset produced by ``hand_in_eye_shooting.ipynb``.
# Expected subfolders: rgb, depth, poses, joints, plus RGB/depth intrinsics.
base_dir = 'data/hand_in_eye8'
rgb_list, depth_list, pose_list, rgb_intrinsics, rgb_coeffs, depth_intrinsics, depth_coeffs, depth_scale, joints_list = read_data(base_dir)

# Choose how to construct ``pose_list`` for the calibration solver.
#
# Required semantic meaning:
#   pose_list[i] == T_hand_to_base for rgb_list[i]
#
# Current notebook behavior:
# - Both conversion lines are commented out, so it uses whatever was loaded
#   from ``poses/`` directly.
# - The script version, ``hand_in_eye_calib.py``, is slightly safer because it
#   automatically prefers joints when ``joints/`` exists.
#
# ViperX recommendation:
# - Prefer joints + FK if you can reliably log ViperX joint states.
# - After implementing ViperX FK, uncomment/adapt the first line below.
# pose_list = [joint_to_hand(joints) for joints in joints_list]
# - Use the second line only if ``poses/`` stores TCP poses that must be mapped
#   into your chosen hand frame.
# pose_list = [tcp_to_hand(pose) for pose in pose_list]

print(f"{len(rgb_list)} poses found")
print(f'Camera matrix: {rgb_intrinsics}')

# Build the exact ChArUco board model used during capture.
# These values must match the printed board dimensions in meters.
charuco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
board = cv2.aruco.CharucoBoard((5, 5), 0.04, 0.03, charuco_dict)

# The calibrator detects target -> camera for each image, combines it with
# hand -> base for each robot pose, and solves the fixed camera -> hand matrix.
calibrator = HandinEyeCalibrator(rgb_intrinsics, rgb_coeffs, charuco_dict, board)
R_cam2gripper_avg, t_cam2gripper_avg = calibrator.perform(rgb_list, pose_list)

print("Average Camera to gripper rotation matrix:")
print(R_cam2gripper_avg)
print("Average Camera to gripper translation vector:")
print(t_cam2gripper_avg)

40 poses found
Camera matrix: [[604.90216064   0.         330.55206299]
 [  0.         605.10333252 241.44529724]
 [  0.           0.           1.        ]]
Average Camera to gripper rotation matrix:
[[-0.24309591 -0.71532616 -0.65514339]
 [ 0.7107732   0.32824672 -0.62213789]
 [ 0.66008018 -0.61689754  0.42863923]]
Average Camera to gripper translation vector:
[[0.43930733]
 [0.3483721 ]
 [0.4645908 ]]


# Visualize results：检查外参方向和点云对齐

这一节不是标定算法本身，而是 sanity check。`R_cam2gripper_avg` 和 `t_cam2gripper_avg` 组成的是 `T_camera_to_hand`。如果矩阵方向用反，后面的点云会明显散开或朝错误方向叠加。

迁移到 ViperX 时，这些可视化代码基本不用改；只要前面得到的每帧 `pose_list` 仍然是 `T_hand_to_base`，那么 `pose_list[i] @ T_camera_to_hand` 就应该得到第 i 帧相机在 base 坐标系下的姿态。

In [5]:
# Visualize the solved camera-to-hand transform as two coordinate frames.
# The original frame is identity. The transformed frame is the camera frame
# placed relative to the hand/gripper frame.
R_avg = R_cam2gripper_avg
t_avg = t_cam2gripper_avg

# Identity pose: a coordinate frame at the hand/gripper origin.
R_original = np.eye(3)
t_original = np.zeros((3, 1))

# Solved relative transform: camera -> gripper/hand.
R_relative = R_avg.copy()
t_relative = t_avg.copy()

# Build 4x4 homogeneous transforms from R/t blocks.
T_original = np.eye(4)
T_original[:3, :3] = R_original
T_original[:3, 3] = t_original.flatten()

T_relative = np.eye(4)
T_relative[:3, :3] = R_relative
T_relative[:3, 3] = t_relative.flatten()

T_transformed = np.dot(T_original, T_relative)

# Open3D coordinate frames are visual aids. If running headless, this may print
# GLFW/OSMesa warnings but does not affect the calibration matrix.
mesh_frame_original = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.6)
mesh_frame_transformed = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.6)

mesh_frame_original.transform(T_original)
mesh_frame_transformed.transform(T_transformed)

o3d.visualization.draw_geometries([mesh_frame_original, mesh_frame_transformed])

[Open3D WARNING] GLFW Error: Failed to detect any supported platform
[Open3D WARNING] GLFW initialized for headless rendering.
[Open3D WARNING] GLFW Error: OSMesa: Library not found
[Open3D WARNING] Failed to create window
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.


In [6]:
# Accumulate a sparse set of RGB-D frames into one point cloud.
# This is a practical validation step: if camera-to-hand and hand-to-base are
# consistent, surfaces observed from different robot poses should overlap.

# Build an Open3D pinhole camera model from RealSense RGB intrinsics.
camera_intrinsic = o3d.camera.PinholeCameraIntrinsic()
camera_intrinsic.set_intrinsics(rgb_list[0].shape[1], rgb_list[0].shape[0],
                                rgb_intrinsics[0, 0], rgb_intrinsics[1, 1],
                                rgb_intrinsics[0, 2], rgb_intrinsics[1, 2])

# Transformation matrix from camera to gripper/hand.
# This is the same output represented as a 4x4 matrix:
#   p_hand = T_cam_to_gripper @ p_camera
T_cam_to_gripper = np.eye(4)
T_cam_to_gripper[0:3, 0:3] = R_cam2gripper_avg
T_cam_to_gripper[0:3, 3] = t_cam2gripper_avg.flatten()

combined_pcd = o3d.geometry.PointCloud()

# Iterate over every 4th image to keep the preview light. Use step 1 for a
# denser debug cloud if performance is acceptable.
for i in range(0,len(rgb_list), 4):
    rgb_img = rgb_list[i]
    depth_img = depth_list[i]

    # ``pose_to_base`` should be T_hand_to_base. This is the key invariant that
    # must stay true after replacing Panda with ViperX.
    pose_to_base = pose_list[i]

    # Create a local camera-frame point cloud from this RGB-D pair.
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(
        o3d.geometry.Image(rgb_img),
        o3d.geometry.Image(depth_img),
        depth_scale=1 / depth_scale,
    )

    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd_image, camera_intrinsic
    )

    # Chain transforms:
    #   T_camera_to_base = T_hand_to_base @ T_camera_to_hand
    # Then move this frame's point cloud from camera coordinates into robot
    # base coordinates.
    cam_to_world = pose_to_base @ T_cam_to_gripper
    pcd.transform(cam_to_world)

    combined_pcd += pcd

# Visualize the combined point cloud
o3d.visualization.draw_geometries([combined_pcd])

[Open3D WARNING] GLFW initialized for headless rendering.
[Open3D WARNING] GLFW Error: OSMesa: Library not found
[Open3D WARNING] Failed to create window
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.


# TSDF reconstruction：用 RGB-D 融合做外参质量检查

TSDF 会把多帧 RGB-D 深度融合成一个粗略 mesh。这里的目的不是替代后续 `reconstruct.py`，而是快速检查手眼外参是否能把不同视角的深度对齐。

如果外参、depth scale 或 `pose_list` 坐标系错了，TSDF 输出通常会出现重影、墙面弯曲、物体重复、或者整体尺度不对。

In [7]:
# Maximum accepted depth in meters. Values farther than this are set to zero
# before integration, which reduces far-background noise.
DEPTH_CUTOFF            = 1

# TSDF voxel size in meters. Smaller voxels preserve detail but cost more
# memory and are more sensitive to calibration noise.
VOXEL_SIZE              =0.005

# Pack camera -> gripper/hand into a 4x4 homogeneous transform.
cam_to_gripper_pose = np.eye(4)
cam_to_gripper_pose[:3, :3] = R_cam2gripper_avg
cam_to_gripper_pose[:3, 3] = t_cam2gripper_avg.squeeze()

# ScalableTSDFVolume incrementally integrates RGB-D observations into a signed
# distance field, then can extract a mesh from it.
volume = o3d.pipelines.integration.ScalableTSDFVolume(
    voxel_length=VOXEL_SIZE,
    sdf_trunc=3 * VOXEL_SIZE,
    color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8
)

for idx in trange(len(rgb_list)):
    # Chain hand pose and camera-to-hand calibration:
    #   T_camera_to_base = T_hand_to_base @ T_camera_to_hand
    # Here ``pose`` names the camera pose in the robot base/world frame.
    pose = pose_list[idx] @ cam_to_gripper_pose

    # Open3D wants contiguous image buffers.
    rgb = rgb_list[idx]
    rgb = np.ascontiguousarray(rgb)

    # Convert raw RealSense depth to meters, then mask far values.
    depth = depth_list[idx] * depth_scale
    depth[depth > DEPTH_CUTOFF] = 0.0 # remove invalid depth
    depth = np.ascontiguousarray(depth.astype(np.float32))

    rgb = o3d.geometry.Image(rgb)
    depth = o3d.geometry.Image(depth)

    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgb, depth, depth_scale=1.0, depth_trunc=4.0, convert_rgb_to_intensity=False)
    intrinsic = camera_intrinsic

    # Open3D TSDF integration expects an extrinsic that maps world/base points
    # into the camera frame, so it uses the inverse of T_camera_to_base.
    # If you accidentally pass ``pose`` directly, the cloud will usually be
    # mirrored/scattered because the transform direction is wrong.
    extrinsic = np.linalg.inv(pose)
    # extrinsic = pose
    volume.integrate(rgbd, intrinsic, extrinsic)

100%|██████████| 40/40 [00:00<00:00, 82.49it/s]


In [25]:
# Extract a mesh from the TSDF volume and visualize a sampled point cloud.
# The PLY is only a debug artifact for calibration quality inspection.
mesh = volume.extract_triangle_mesh()
sampled_pcd = mesh.sample_points_uniformly(number_of_points=100000)
o3d.visualization.draw_geometries([sampled_pcd])
o3d.io.write_triangle_mesh("pointcloud.ply", mesh)

[Open3D WARNING] GLFW Error: X11: The DISPLAY environment variable is missing
[Open3D WARNING] Failed to initialize GLFW
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.


True

In [26]:
# Scratch/import cell kept from the original notebook.
# roboticstoolbox is already imported above; this line is harmless but not
# required for the calibration flow.
import roboticstoolbox as rtb


[(4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4), (4, 4)]


In [27]:
# Scratch cell from the original notebook. The saved output below is stale
# notebook state, not part of the reproducible calibration pipeline.


array([[ 0.01495648, -0.99906156, -0.04065007,  0.06511291],
       [ 0.99819093,  0.01728653, -0.05758606, -0.0088448 ],
       [ 0.05823475, -0.03971527,  0.99751267,  0.01098037],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])